In [1]:
# data_transform_fast.py
"""
Fast dataset transform for ShieldPatch:
 - Vectorized text cleaning (no NLTK)
 - Numeric fixes, cvss imputation, severity derive
 - OS flags, os_count
 - Drops rows with missing target / too-short text

Much faster than the heavy NLTK version.
"""

import os
import re
import pandas as pd
import numpy as np
from datetime import datetime

INPUT_PATH = "../data/ml_dataset.csv"
OUTPUT_PATH = "../data/ml_dataset_cleaned_fast.csv"
REF_DATE = pd.to_datetime("2025-11-27")

# small stopword list (fast, inline) — extend if you like
SMALL_STOPWORDS = {
    "the","and","or","of","in","on","for","to","with","a","an","is","are","this","that",
    "by","from","was","were","be","as","it","its","which","at","version","prior","product",
    "v","v.", "update", "updates", "released"
}

# Pre-compiled regex (fast)
URL_RE = re.compile(r'http\S+|www\.\S+')
HTML_RE = re.compile(r'<[^>]+>')
NON_ALNUM_RE = re.compile(r'[^a-z0-9\s]')
MULTI_SPACE_RE = re.compile(r'\s+')

def fast_clean_text_series(series: pd.Series) -> pd.Series:
    """
    Vectorized cleaning:
    - lower
    - remove URLs, HTML
    - keep only alnum and spaces
    - collapse spaces
    - simple stopword removal using split+join (still python-level per-row, but fast enough)
    """
    s = series.fillna("").astype(str)
    s = s.str.lower()
    s = s.str.replace(URL_RE, " ", regex=True)
    s = s.str.replace(HTML_RE, " ", regex=True)
    s = s.str.replace(NON_ALNUM_RE, " ", regex=True)
    s = s.str.replace(MULTI_SPACE_RE, " ", regex=True).str.strip()

    # Remove very short strings quickly
    s = s.where(s.str.len() > 0, "")

    # Remove small stopwords: apply a lightweight filter (fast for ~2k rows)
    def remove_small_stopwords(text):
        if not text:
            return ""
        tokens = text.split()
        out_tokens = [t for t in tokens if t not in SMALL_STOPWORDS and len(t) > 1 and not t.isdigit()]
        return " ".join(out_tokens)

    return s.map(remove_small_stopwords)

def parse_datetime_safe(s):
    try:
        return pd.to_datetime(s, errors='coerce')
    except Exception:
        return pd.NaT

def compute_years_since(published_ts, ref_date=REF_DATE):
    if pd.isna(published_ts):
        return np.nan
    delta_days = (ref_date - published_ts).days
    return float(delta_days) / 365.25

def severity_from_cvss(cvss_val):
    try:
        cvss = float(cvss_val)
    except Exception:
        return np.nan
    if cvss <= 3.9:
        return "LOW"
    if cvss <= 6.9:
        return "MEDIUM"
    if cvss <= 8.9:
        return "HIGH"
    return "CRITICAL"

def transform(input_path=INPUT_PATH, output_path=OUTPUT_PATH):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    df = pd.read_csv(input_path)
    print("Loaded:", input_path, "shape:", df.shape)

    # drop full duplicates & by cve_id
    df.drop_duplicates(inplace=True)
    if 'cve_id' in df.columns:
        df.drop_duplicates(subset=['cve_id'], inplace=True)

    # published -> datetime (vectorized)
    df['published_dt'] = pd.to_datetime(df['published'], errors='coerce')

    # years_since_published: prefer numeric existing, else compute from published_dt, else median
    df['years_since_published_fixed'] = pd.to_numeric(df.get('years_since_published', np.nan), errors='coerce')
    mask = df['years_since_published_fixed'].isna() & df['published_dt'].notna()
    df.loc[mask, 'years_since_published_fixed'] = (REF_DATE - df.loc[mask, 'published_dt']).dt.days / 365.25
    df['years_since_published_fixed'].fillna(df['years_since_published_fixed'].median(skipna=True), inplace=True)

    # cvss_score numeric + median impute (vectorized)
    df['cvss_score_fixed'] = pd.to_numeric(df.get('cvss_score', np.nan), errors='coerce').fillna(df['cvss_score'].median(skipna=True))
    # if still na (rare), fill with 6.0
    df['cvss_score_fixed'].fillna(6.0, inplace=True)

    # severity standardize (vectorized apply)
    if 'severity' in df.columns:
        df['severity_fixed'] = df['severity'].astype(str).str.strip().str.upper()
        df['severity_fixed'] = df['severity_fixed'].where(df['severity_fixed'].isin(['LOW','MEDIUM','HIGH','CRITICAL']), np.nan)
    else:
        df['severity_fixed'] = np.nan
    df['severity_fixed'].fillna(df['cvss_score_fixed'].apply(severity_from_cvss), inplace=True)

    # references_count, weaknesses_count numeric
    df['references_count_fixed'] = pd.to_numeric(df.get('references_count', 0), errors='coerce').fillna(0).astype(int)
    df['weaknesses_count_fixed'] = pd.to_numeric(df.get('weaknesses_count', 0), errors='coerce').fillna(0).astype(int)

    # OS flags: vectorized ensure 0/1
    for col in ['has_windows', 'has_linux', 'has_android']:
        if col in df.columns:
            df[col + "_fixed"] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int).clip(0,1)
        else:
            df[col + "_fixed"] = 0

    # os_count
    df['os_count'] = df[['has_windows_fixed', 'has_linux_fixed', 'has_android_fixed']].sum(axis=1).astype(int)

    # Fast text cleaning vectorized
    if 'description_text' in df.columns:
        df['description_text_clean'] = fast_clean_text_series(df['description_text'])
    else:
        df['description_text_clean'] = fast_clean_text_series(df.astype(str).sum(axis=1))

    # ensure target exists and numeric
    if 'risk_score' not in df.columns:
        raise KeyError("Target column 'risk_score' not found.")
    df['risk_score_fixed'] = pd.to_numeric(df['risk_score'], errors='coerce')

    # drop missing target
    df = df[df['risk_score_fixed'].notna()].copy()

    # drop too-short descriptions
    df = df[df['description_text_clean'].str.len() > 5].copy()

    # select and rename
    out = df[[
        'cve_id',
        'published_dt',
        'years_since_published_fixed',
        'cvss_score_fixed',
        'severity_fixed',
        'references_count_fixed',
        'weaknesses_count_fixed',
        'has_windows_fixed',
        'has_linux_fixed',
        'has_android_fixed',
        'os_count',
        'description_text_clean',
        'risk_score_fixed'
    ]].copy()

    out = out.rename(columns={
        'published_dt': 'published',
        'years_since_published_fixed': 'years_since_published',
        'cvss_score_fixed': 'cvss_score',
        'severity_fixed': 'severity',
        'references_count_fixed': 'references_count',
        'weaknesses_count_fixed': 'weaknesses_count',
        'has_windows_fixed': 'has_windows',
        'has_linux_fixed': 'has_linux',
        'has_android_fixed': 'has_android',
        'description_text_clean': 'description_text',
        'risk_score_fixed': 'risk_score'
    })

    out.to_csv(output_path, index=False)
    print("Saved cleaned (fast) dataset to:", output_path)
    print("Cleaned shape:", out.shape)
    print(out.head().to_string(index=False))

if __name__ == "__main__":
    transform()

Loaded: ../data/ml_dataset.csv shape: (2428, 12)
Saved cleaned (fast) dataset to: ../data/ml_dataset_cleaned_fast.csv
Cleaned shape: (2428, 13)
        cve_id               published  years_since_published  cvss_score severity  references_count  weaknesses_count  has_windows  has_linux  has_android  os_count                                                                                                                                                                                                                                                                                                                                                                                                description_text  risk_score
CVE-2005-10004 2025-08-30 14:15:32.040                   0.24         8.7     HIGH                 6                 1            0          0            0         0                                                   cacti versions contain remote command execution vulnerability 

/var/folders/3m/gfvpvdkx01lckx4s_8z9qvjw0000gn/T/ipykernel_9991/3868773512.py:108: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['years_since_published_fixed'].fillna(df['years_since_published_fixed'].median(skipna=True), inplace=True)
/var/folders/3m/gfvpvdkx01lckx4s_8z9qvjw0000gn/T/ipykernel_9991/3868773512.py:113: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work beca

In [3]:
# verify_dataset.py
import os
import json
import pandas as pd
import numpy as np
from collections import OrderedDict

CSV_PATH = "../data/ml_dataset_cleaned_fast.csv"    # ★ Your actual dataset path
REPORT_PATH = "../data/verification_report.json"     # ★ JSON report output

REQUIRED_COLUMNS = [
    'cve_id', 'published', 'years_since_published', 'cvss_score', 'severity',
    'references_count', 'weaknesses_count', 'has_windows', 'has_linux',
    'has_android', 'os_count', 'description_text', 'risk_score'
]

ALLOWED_SEVERITY = {'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'}

def short_row_preview(df, idxs, n=5):
    if len(idxs) == 0:
        return []
    preview = []
    for i in idxs[:n]:
        preview.append(df.iloc[i].to_dict())
    return preview

def verify_dataset():
    report = OrderedDict()

    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"File not found: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH)

    report["file"] = os.path.abspath(CSV_PATH)
    report["shape"] = df.shape
    report["head"] = df.head(3).to_dict(orient="records")

    # Column presence
    present_cols = list(df.columns)
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in present_cols]
    report["missing_required_columns"] = missing_cols

    # Duplicates
    report["duplicate_all_count"] = int(df.duplicated().sum())
    if "cve_id" in df.columns:
        report["duplicate_cve_id_count"] = int(df.duplicated(subset=["cve_id"]).sum())
    else:
        report["duplicate_cve_id_count"] = None

    # Missing values
    report["missing_values"] = df.isna().sum().to_dict()

    # Numeric dtype check
    numeric_cols = [
        "years_since_published", "cvss_score", "references_count",
        "weaknesses_count", "has_windows", "has_linux", "has_android",
        "os_count", "risk_score"
    ]

    numeric_status = {}
    for col in numeric_cols:
        if col in df.columns:
            coerced = pd.to_numeric(df[col], errors="coerce")
            numeric_status[col] = {
                "dtype": str(df[col].dtype),
                "non_numeric_count": int((coerced.isna() & df[col].notna()).sum())
            }
        else:
            numeric_status[col] = {"dtype": "missing", "non_numeric_count": None}
    report["numeric_checks"] = numeric_status

    # CVSS sanity
    if "cvss_score" in df.columns:
        cv = pd.to_numeric(df["cvss_score"], errors="coerce")
        report["cvss_range"] = {
            "min": float(cv.min()) if cv.notna().any() else None,
            "max": float(cv.max()) if cv.notna().any() else None,
            "out_of_range_count": int(((cv < 0) | (cv > 10)).sum())
        }

    # Severity check
    if "severity" in df.columns:
        unique_sev = sorted(df["severity"].astype(str).unique())
        invalid = [s for s in unique_sev if s not in ALLOWED_SEVERITY]
        report["severity_unique"] = unique_sev
        report["severity_invalid"] = invalid

    # OS flags
    os_info = {}
    for col in ["has_windows", "has_linux", "has_android"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            uniq = sorted(vals.dropna().unique().tolist())
            os_info[col] = {
                "unique_values": uniq,
                "non_binary_count": int(((vals != 0) & (vals != 1)).sum())
            }
        else:
            os_info[col] = {"unique_values": None, "non_binary_count": None}
    report["os_flags"] = os_info

    # Description text
    desc = df["description_text"].fillna("").astype(str)
    lengths = desc.str.len()
    report["description_text_stats"] = {
        "min_length": int(lengths.min()),
        "median_length": float(lengths.median()),
        "max_length": int(lengths.max()),
        "short_count_<=5": int((lengths <= 5).sum())
    }

    # Risk score
    r = pd.to_numeric(df["risk_score"], errors="coerce")
    report["risk_score_stats"] = {
        "min": float(r.min()),
        "max": float(r.max()),
        "mean": float(r.mean()),
        "median": float(r.median()),
        "missing_count": int(r.isna().sum())
    }

    # Save JSON report
    with open(REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)

    # Console summary
    print("\n==== DATASET VERIFICATION COMPLETE ====")
    print("File:", report["file"])
    print("Shape:", report["shape"])
    print("Missing required columns:", report["missing_required_columns"])
    print("Missing values:", report["missing_values"])
    print("Numeric column issues:", report["numeric_checks"])
    print("Severity unique:", report.get("severity_unique"))
    print("OS flags:", report["os_flags"])
    print("Risk score:", report["risk_score_stats"])
    print("\nJSON report saved to:", REPORT_PATH)

    return report


if __name__ == "__main__":
    verify_dataset()


==== DATASET VERIFICATION COMPLETE ====
File: /Users/ajaykumar/Desktop/ShieldPatch/ML_MODEL_DEV/data/ml_dataset_cleaned_fast.csv
Shape: (2428, 13)
Missing required columns: []
Missing values: {'cve_id': 0, 'published': 1314, 'years_since_published': 0, 'cvss_score': 0, 'severity': 0, 'references_count': 0, 'weaknesses_count': 0, 'has_windows': 0, 'has_linux': 0, 'has_android': 0, 'os_count': 0, 'description_text': 0, 'risk_score': 0}
Numeric column issues: {'years_since_published': {'dtype': 'float64', 'non_numeric_count': 0}, 'cvss_score': {'dtype': 'float64', 'non_numeric_count': 0}, 'references_count': {'dtype': 'int64', 'non_numeric_count': 0}, 'weaknesses_count': {'dtype': 'int64', 'non_numeric_count': 0}, 'has_windows': {'dtype': 'int64', 'non_numeric_count': 0}, 'has_linux': {'dtype': 'int64', 'non_numeric_count': 0}, 'has_android': {'dtype': 'int64', 'non_numeric_count': 0}, 'os_count': {'dtype': 'int64', 'non_numeric_count': 0}, 'risk_score': {'dtype': 'float64', 'non_numeric